In [2]:
!unzip transformer-domain-shift.zip
%cd transformer-domain-shift

Archive:  transformer-domain-shift.zip
   creating: transformer-domain-shift/
   creating: transformer-domain-shift/data/
  inflating: transformer-domain-shift/data/load_dataset.py  
   creating: transformer-domain-shift/eval/
  inflating: transformer-domain-shift/eval/evaluate_model.py  
   creating: transformer-domain-shift/experiments/
  inflating: transformer-domain-shift/experiments/confidence_analysis.py  
  inflating: transformer-domain-shift/experiments/error_analysis.py  
   creating: transformer-domain-shift/models/
   creating: transformer-domain-shift/notebooks/
   creating: transformer-domain-shift/README.md/
  inflating: transformer-domain-shift/requirements.txt  
   creating: transformer-domain-shift/results/
   creating: transformer-domain-shift/train/
  inflating: transformer-domain-shift/train/train_bert.py  
/content/transformer-domain-shift


In [3]:
pip install -r requirements.txt

In [1]:
import torch
print(torch.cuda.is_available())

True


In [5]:
!python data/load_dataset.py

tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 174kB/s]
vocab.txt: 232kB [00:00, 3.10MB/s]
tokenizer.json: 466kB [00:00, 9.05MB/s]
README.md: 7.81kB [00:00, 28.2MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:01<00:00, 19.0MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 25.5MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:01<00:00, 34.9MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 122578.39 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 202489.95 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 125639.20 examples/s]
Map: 100% 25000/25000 [00:22<00:00, 1105.88 examples/s]
Map: 100% 25000/25000 [00:22<00:00, 1126.49 examples/s]
Map: 100% 50000/50000 [00:44<00:00, 1133.03 examples/s]
README.md: 35.3kB [00:00, 83.6MB/s]
sst2/train-00000-of-00001.parquet: 100% 3.11M/3.11M [00:02<00:00, 1.20MB/s]
sst2/validation-00000-of-00001.parquet: 100% 72.8k/72.8k [00:0

In [7]:
!python train/train_bert.py

Traceback (most recent call last):
  File "/content/transformer-domain-shift/train/train_bert.py", line 2, in <module>
    from transformers import BertForSequenceClassification, AdamW
ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)


In [19]:
%%writefile train/train_bert.py
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForSequenceClassification, BertTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
import time
import datetime
import random
import os
import sys

# Add the current directory to sys.path to find local modules
sys.path.insert(0, os.path.abspath('.'))

from data.load_dataset import load_imdb_dataset

# Set the seed for reproducibility
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 3
LEARNING_RATE = 2e-5

MODEL_NAME = "bert-base-uncased"

SAVE_PATH = "results/bert_imdb.pt"

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round((elapsed)))))

def train():

    train_loader, test_loader = load_imdb_dataset()

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    model.to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    model.train()

    for epoch in range(EPOCHS):

        total_loss = 0

        progress_bar = tqdm(train_loader)

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            progress_bar.set_description(
                f"Epoch {epoch+1} Loss {loss.item():.4f}" # Fixed: Added closing double quote and parenthesis
            )

        avg_loss = total_loss / len(train_loader)

        print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

    os.makedirs("results", exist_ok=True)

    torch.save(model.state_dict(), SAVE_PATH)

    print("\nModel saved to", SAVE_PATH)


if __name__ == "__main__":

    train()


Overwriting train/train_bert.py


In [20]:
!python train/train_bert.py

Map: 100% 25000/25000 [00:22<00:00, 1093.88 examples/s]
Map: 100% 25000/25000 [00:21<00:00, 1161.88 examples/s]
Map: 100% 50000/50000 [00:45<00:00, 1105.06 examples/s]
config.json: 100% 570/570 [00:00<00:00, 2.96MB/s]
model.safetensors: 100% 440M/440M [00:03<00:00, 146MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 936.64it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias           

In [23]:
!python eval/evaluate_model.py


Loading datasets...
Map: 100% 67349/67349 [00:08<00:00, 8254.98 examples/s]
Map: 100% 872/872 [00:00<00:00, 4568.56 examples/s]
Map: 100% 1821/1821 [00:00<00:00, 4257.03 examples/s]
Loading trained model...
Loading weights: 100% 199/199 [00:00<00:00, 1093.82it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                

In [22]:
%%writefile eval/evaluate_model.py
import torch
from transformers import BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
import sys
import os

# Add the current directory to sys.path to find local modules
sys.path.insert(0, os.path.abspath('.'))

from data.load_dataset import load_imdb_dataset, load_sst2_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "results/bert_imdb.pt"
MODEL_NAME = "bert-base-uncased"

def evaluate(model, dataloader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    cm = confusion_matrix(all_labels, all_preds)

    return acc, f1, cm


def main():

    print("\nLoading datasets...")

    # load_imdb_dataset returns train_loader, test_loader
    _, imdb_test = load_imdb_dataset()
    sst2_test = load_sst2_dataset()

    print("Loading trained model...")

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"Model loaded from {MODEL_PATH}")
    else:
        print(f"Error: Model file not found at {MODEL_PATH}. Please ensure the training script ran successfully and saved the model.")
        return # Exit if model not found

    model.to(DEVICE)

    print("\nEvaluating on IMDB (In-Domain)...")

    imdb_acc, imdb_f1, imdb_cm = evaluate(model, imdb_test)

    print("\nEvaluating on SST-2 (Domain Shift)...")

    sst2_acc, sst2_f1, sst2_cm = evaluate(model, sst2_test)

    print("\n===== RESULTS ===")

    print("\nIMDB Test Results")
    print("Accuracy:", imdb_acc)
    print("F1 Score:", imdb_f1)
    print("Confusion Matrix:\n", imdb_cm)

    print("\nSST-2 Test Results")
    print("Accuracy:", sst2_acc)
    print("F1 Score:", sst2_f1)
    print("Confusion Matrix:\n", sst2_cm)


if __name__ == "__main__":
    main()


Overwriting eval/evaluate_model.py


In [26]:
!python experiments/confidence_analysis.py

Loading datasets...
Map: 100% 872/872 [00:00<00:00, 6885.76 examples/s]
Loading model...
Loading weights: 100% 199/199 [00:00<00:00, 885.90it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

In [25]:
%%writefile experiments/confidence_analysis.py
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import BertForSequenceClassification
from sklearn.metrics import accuracy_score
import sys
import os

# Add the current directory to sys.path to find local modules
sys.path.insert(0, os.path.abspath('.'))

from data.load_dataset import load_imdb_dataset, load_sst2_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "results/bert_imdb.pt"
MODEL_NAME = "bert-base-uncased"

BINS = 10


def get_predictions(model, dataloader):

    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probs = torch.softmax(outputs.logits, dim=1)

            conf, preds = torch.max(probs, dim=1)

            all_probs.extend(conf.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_probs), np.array(all_preds), np.array(all_labels)


def compute_ece(confidences, preds, labels):

    bin_boundaries = np.linspace(0, 1, BINS + 1)

    ece = 0

    for i in range(BINS):

        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:

            accuracy = np.mean(preds[in_bin] == labels[in_bin])
            avg_conf = np.mean(confidences[in_bin])

            ece += np.abs(avg_conf - accuracy) * prop_in_bin

    return ece


def plot_confidence_hist(confidences, title, save_path):

    plt.figure()

    plt.hist(confidences, bins=10)

    plt.title(title)
    plt.xlabel("Confidence")
    plt.ylabel("Frequency")

    plt.savefig(save_path)
    plt.close()


def reliability_diagram(confidences, preds, labels, title, save_path):

    bin_boundaries = np.linspace(0, 1, BINS + 1)

    accs = []
    confs = []

    for i in range(BINS):

        lower = bin_boundaries[i]
        upper = bin_boundaries[i + 1]

        mask = (confidences > lower) & (confidences <= upper)

        if np.sum(mask) > 0:

            acc = np.mean(preds[mask] == labels[mask])
            conf = np.mean(confidences[mask])

        else:

            acc = 0
            conf = 0

        accs.append(acc)
        confs.append(conf)

    plt.figure()

    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect Calibration")

    plt.plot(confs, accs, marker="o", label="Model")

    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)

    plt.legend()

    plt.savefig(save_path)
    plt.close()


def main():

    print("Loading datasets...")

    imdb_train, imdb_test = load_imdb_dataset()
    sst2_test = load_sst2_dataset()

    print("Loading model...")

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"Model loaded from {MODEL_PATH}")
    else:
        print(f"Error: Model file not found at {MODEL_PATH}. Please ensure the training script ran successfully and saved the model.")
        return # Exit if model not found

    model.to(DEVICE)

    print("Running IMDB confidence analysis...")

    imdb_conf, imdb_preds, imdb_labels = get_predictions(model, imdb_test)

    imdb_ece = compute_ece(imdb_conf, imdb_preds, imdb_labels)

    print("IMDB Accuracy:", accuracy_score(imdb_labels, imdb_preds))
    print("IMDB ECE:", imdb_ece)

    os.makedirs("results", exist_ok=True)
    plot_confidence_hist(
        imdb_conf,
        "Confidence Distribution (IMDB)",
        "results/imdb_confidence_hist.png"
    )

    reliability_diagram(
        imdb_conf,
        imdb_preds,
        imdb_labels,
        "Reliability Diagram (IMDB)",
        "results/imdb_reliability.png"
    )

    print("Running SST-2 confidence analysis...")

    sst_conf, sst_preds, sst_labels = get_predictions(model, sst2_test)

    sst_ece = compute_ece(sst_conf, sst_preds, sst_labels)

    print("SST2 Accuracy:", accuracy_score(sst_labels, sst_preds))
    print("SST2 ECE:", sst_ece)

    plot_confidence_hist(
        sst_conf,
        "Confidence Distribution (SST2)",
        "results/sst2_confidence_hist.png"
    )

    reliability_diagram(
        sst_conf,
        sst_preds,
        sst_labels,
        "Reliability Diagram (SST2)",
        "results/sst2_reliability.png"
    )


if __name__ == "__main__":
    main()

Overwriting experiments/confidence_analysis.py


In [27]:
!python experiments/error_analysis.py

Loading weights: 100% 199/199 [00:00<00:00, 920.95it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing

In [29]:
!zip -r /content/transformer-domain-shift /content/transformer-domain-shift

updating: content/transformer-domain-shift/ (stored 0%)
updating: content/transformer-domain-shift/README.md/ (stored 0%)
updating: content/transformer-domain-shift/requirements.txt (deflated 18%)
updating: content/transformer-domain-shift/eval/ (stored 0%)
updating: content/transformer-domain-shift/eval/evaluate_model.py (deflated 61%)
updating: content/transformer-domain-shift/models/ (stored 0%)
updating: content/transformer-domain-shift/experiments/ (stored 0%)
updating: content/transformer-domain-shift/experiments/confidence_analysis.py (deflated 68%)
updating: content/transformer-domain-shift/experiments/error_analysis.py (deflated 59%)
updating: content/transformer-domain-shift/train/ (stored 0%)
updating: content/transformer-domain-shift/train/train_bert.py (deflated 57%)
updating: content/transformer-domain-shift/results/ (stored 0%)
updating: content/transformer-domain-shift/results/bert_imdb.pt (deflated 7%)
updating: content/transformer-domain-shift/results/imdb_reliability

In [30]:
from google.colab import files
files.download("/content/transformer-domain-shift.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>